In [ ]:
#数据探索部分

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['font.sans-serif'] = ['SimHei']
plt.rcParams['axes.unicode_minus'] = False  

os.makedirs('数据探索图片', exist_ok=True)

try:
    with open('used_car_train_20200313.csv', 'r', encoding='utf-8') as f:
        lines = f.readlines()

    header = lines[0].strip()
    columns = header.split()

    data = []
    skipped_lines = 0
    for i, line in enumerate(lines[1:]):
        if not line.strip():
            continue

        row_data = line.strip().split()

        if len(row_data) >= len(columns):
            data.append(row_data[:len(columns)])
        else:
            skipped_lines += 1
            if skipped_lines <= 10: 
                print(f"第{i+1}行数据不完整，跳过")
    
    if skipped_lines > 10:
        print(f"... 还有 {skipped_lines - 10} 行被跳过")
    
    train_data = pd.DataFrame(data, columns=columns)
    
    print("训练集形状:", train_data.shape)
    
    with open('used_car_testB_20200421.csv', 'r', encoding='utf-8') as f:
        lines = f.readlines()
    
    header = lines[0].strip()
    columns = header.split()
    
    data = []
    skipped_lines_test = 0
    for i, line in enumerate(lines[1:]):
        if not line.strip():
            continue
            
        row_data = line.strip().split()
        if len(row_data) >= len(columns):
            data.append(row_data[:len(columns)])
        else:
            skipped_lines_test += 1
    
    test_data = pd.DataFrame(data, columns=columns)
    
    print(f"跳过 {skipped_lines_test} 行不完整数据")
    print("测试集形状:", test_data.shape)
    
except Exception as e:
    print(f"处理数据时出错: {e}")

numeric_columns = ['SaleID', 'regDate', 'model', 'brand', 'bodyType', 'fuelType', 
                  'gearbox', 'power', 'kilometer', 'notRepairedDamage', 'regionCode', 
                  'seller', 'offerType', 'creatDate', 'price'] + [f'v_{i}' for i in range(15)]

for col in numeric_columns:
    if col in train_data.columns:
        train_data[col] = train_data[col].replace('-', np.nan)
        train_data[col] = pd.to_numeric(train_data[col], errors='coerce')
        
        if col in test_data.columns:
            test_data[col] = test_data[col].replace('-', np.nan)
            test_data[col] = pd.to_numeric(test_data[col], errors='coerce')

print("\n数据基本信息:")
print(train_data.info())


plt.figure(figsize=(15, 10))

# 原始价格分布
plt.subplot(2, 3, 1)
plt.hist(train_data['price'], bins=50, alpha=0.7, color='skyblue', edgecolor='black')
plt.title('二手车价格分布')
plt.xlabel('价格')
plt.ylabel('频次')
plt.xlim(0, 60000)
plt.grid(True, alpha=0.3)

# 价格箱线图
plt.subplot(2, 3, 2)
plt.boxplot(train_data['price'])
plt.title('价格箱线图')
plt.ylabel('价格')
plt.grid(True, alpha=0.3)

# 对数价格分布
plt.subplot(2, 3, 3)
log_price = np.log1p(train_data['price'])
plt.hist(log_price, bins=50, alpha=0.7, color='lightgreen', edgecolor='black')
plt.title('对数价格分布')
plt.xlabel('log(价格+1)')
plt.ylabel('频次')
plt.grid(True, alpha=0.3)

# 价格百分位统计
plt.subplot(2, 3, 4)
price_percentiles = [0, 0.1, 0.25, 0.5, 0.75, 0.9, 0.95, 0.99, 1]
price_stats = train_data['price'].quantile(price_percentiles)
plt.bar(range(len(price_stats)), price_stats.values, alpha=0.7, color='coral')
plt.title('价格百分位统计')
plt.xticks(range(len(price_stats)), [f'{p*100}%' for p in price_percentiles], rotation=45)
plt.ylabel('价格')
plt.grid(True, alpha=0.3)

# 价格与功率的关系
plt.subplot(2, 3, 5)
plt.scatter(train_data['power'], train_data['price'], alpha=0.3, s=1)
plt.title('发动机功率与价格关系')
plt.xlabel('功率')
plt.ylabel('价格')
plt.grid(True, alpha=0.3)

# 价格与行驶里程的关系
plt.subplot(2, 3, 6)
plt.scatter(train_data['kilometer'], train_data['price'], alpha=0.3, s=1)
plt.title('行驶里程与价格关系')
plt.xlabel('行驶里程(万km)')
plt.ylabel('价格')
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('数据探索图片/price_analysis.png', dpi=300, bbox_inches='tight')
plt.show()

print("\n价格统计描述:")
price_desc = train_data['price'].describe(percentiles=[0.01, 0.05, 0.1, 0.25, 0.5, 0.75, 0.9, 0.95, 0.99])
print(price_desc)

key_numerical = ['power', 'kilometer', 'model', 'brand', 'v_0', 'v_1', 'v_2']

plt.figure(figsize=(20, 15))
for i, feature in enumerate(key_numerical, 1):
    plt.subplot(3, 3, i)
    plt.hist(train_data[feature], bins=50, alpha=0.7, color='lightblue', edgecolor='black')
    plt.title(f'{feature}分布')
    if feature == 'power':
        plt.xlim(0, 1000)
        x_ticks = np.arange(0, 1001, 50)
        plt.xticks(x_ticks)
    plt.xlabel(feature)
    plt.ylabel('频次')
    plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('数据探索图片/numerical_features.png', dpi=300, bbox_inches='tight')
plt.show()

categorical_features = ['bodyType', 'fuelType', 'gearbox', 'notRepairedDamage', 'seller', 'offerType']

plt.figure(figsize=(20, 12))
for i, feature in enumerate(categorical_features, 1):
    plt.subplot(2, 3, i)
    value_counts = train_data[feature].value_counts().sort_index()
    
    # 为类别特征标签
    if feature == 'bodyType':
        labels = ['豪华轿车', '微型车', '厢型车', '大巴车', '敞篷车', '双门汽车', '商务车', '搅拌车']
    elif feature == 'fuelType':
        labels = ['汽油', '柴油', '液化石油气', '天然气', '混合动力', '其他', '电动']
    elif feature == 'gearbox':
        labels = ['手动', '自动']
    elif feature == 'notRepairedDamage':
        labels = ['有损坏', '无损坏']
    elif feature == 'seller':
        labels = ['个体', '非个体']
    elif feature == 'offerType':
        labels = ['提供', '请求']
    else:
        labels = [str(x) for x in value_counts.index]
    
    existing_labels = [labels[int(idx)] for idx in value_counts.index if int(idx) < len(labels)]
    
    plt.bar(range(len(value_counts)), value_counts.values, alpha=0.7, color='lightcoral', edgecolor='black')
    plt.title(f'{feature}分布')
    plt.xlabel(feature)
    plt.ylabel('数量')
    plt.xticks(range(len(value_counts)), existing_labels, rotation=45)
    plt.grid(True, alpha=0.3)

    for j, v in enumerate(value_counts.values):
        plt.text(j, v, str(v), ha='center', va='bottom', fontsize=8)
plt.tight_layout()
plt.savefig('数据探索图片/categorical_features.png', dpi=300, bbox_inches='tight')
plt.show()

missing_data = train_data.isnull().sum()
missing_percent = (missing_data / len(train_data)) * 100
missing_info = pd.DataFrame({'缺失数量': missing_data, '缺失比例%': missing_percent})
missing_info = missing_info[missing_info['缺失数量'] > 0].sort_values('缺失数量', ascending=False)

print("缺失值统计:")
print(missing_info)

plt.figure(figsize=(6, 3))
missing_info = missing_info.sort_values('缺失比例%', ascending=True)
plt.barh(missing_info.index, missing_info['缺失比例%'], color='#e67e22', alpha=0.8)
plt.title('各字段缺失值比例', fontsize=14, fontweight='bold')
plt.xlabel('缺失比例 (%)', fontsize=12)
plt.grid(axis='x', alpha=0.3)
for i, v in enumerate(missing_info['缺失比例%']):
    plt.text(v + 0.5, i, f'{v:.1f}%', va='center', fontsize=10)
plt.tight_layout()
plt.savefig('数据探索图片/missing_values.png', dpi=300, bbox_inches='tight')
plt.show()

plt.figure(figsize=(12, 6))

plt.subplot(1, 2, 1)
plt.boxplot(train_data['power'])
plt.title('发动机功率箱线图', fontsize=12)
plt.ylabel('功率')
plt.grid(True, alpha=0.3)

plt.subplot(1, 2, 2)
# 显示合理功率范围内的分布
reasonable_power = train_data[(train_data['power'] > 0) & (train_data['power'] <= 600)]
plt.hist(reasonable_power['power'], bins=50, alpha=0.7, color='skyblue', edgecolor='black')
plt.title('合理功率分布 (0-600)', fontsize=12)
plt.xlabel('功率')
plt.ylabel('频次')
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('数据探索图片/outlier_detection.png', dpi=300, bbox_inches='tight')
plt.show()

# 功率异常值统计
power_outliers = train_data[train_data['power'] > 600]
print(f"\n功率异常值数量（>600）: {len(power_outliers)}")
print(f"功率异常值比例: {len(power_outliers)/len(train_data)*100:.2f}%")

# 进行相关性分析
correlation_features = ['power', 'kilometer', 'model', 'brand', 'bodyType', 'fuelType', 
                       'gearbox', 'price'] + [f'v_{i}' for i in range(5)]

# 确保所有特征都存在
correlation_features = [f for f in correlation_features if f in train_data.columns]

corr_matrix = train_data[correlation_features].corr()

plt.figure(figsize=(8,6))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))  # 创建上三角掩码
sns.heatmap(corr_matrix, mask=mask, annot=True, fmt='.2f', cmap='coolwarm', center=0,
            square=True, linewidths=0.5, cbar_kws={"shrink": .8})
plt.title('特征相关性热力图')
plt.tight_layout()
plt.savefig('数据探索图片/correlation_heatmap.png', dpi=300, bbox_inches='tight')
plt.show()

# 价格与主要特征的相关性
price_corr = corr_matrix['price'].sort_values(ascending=False)
print("\n价格相关性排序（绝对值前10名）:")
price_corr_abs = corr_matrix['price'].abs().sort_values(ascending=False)
print(price_corr_abs.head(10))

train_data.to_csv('train_data_raw.csv', index=False)
test_data.to_csv('test_data_raw.csv', index=False)

In [ ]:
# 第二步：数据预处理

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.decomposition import PCA
from sklearn.impute import SimpleImputer
import warnings
import os
warnings.filterwarnings('ignore')

plt.rcParams['font.sans-serif'] = ['SimHei']
plt.rcParams['axes.unicode_minus'] = False  

os.makedirs('预处理图片', exist_ok=True)

train_data = pd.read_csv('train_data_raw.csv')
test_data = pd.read_csv('test_data_raw.csv')

print("原始数据形状:", train_data.shape, test_data.shape)

# 转换时间格式
train_data['regDate'] = pd.to_datetime(train_data['regDate'], format='%Y%m%d', errors='coerce')
train_data['creatDate'] = pd.to_datetime(train_data['creatDate'], format='%Y%m%d', errors='coerce')
test_data['regDate'] = pd.to_datetime(test_data['regDate'], format='%Y%m%d', errors='coerce')
test_data['creatDate'] = pd.to_datetime(test_data['creatDate'], format='%Y%m%d', errors='coerce')

# 计算车龄（年）
train_data['car_age'] = (train_data['creatDate'] - train_data['regDate']).dt.days / 365.25
test_data['car_age'] = (test_data['creatDate'] - test_data['regDate']).dt.days / 365.25

# 从日期中提取年份和月份
train_data['regYear'] = train_data['regDate'].dt.year
train_data['regMonth'] = train_data['regDate'].dt.month
train_data['creatYear'] = train_data['creatDate'].dt.year
train_data['creatMonth'] = train_data['creatDate'].dt.month

test_data['regYear'] = test_data['regDate'].dt.year
test_data['regMonth'] = test_data['regDate'].dt.month
test_data['creatYear'] = test_data['creatDate'].dt.year
test_data['creatMonth'] = test_data['creatDate'].dt.month



# 记录处理前的数据量
original_size = len(train_data)

# 功率异常值处理
train_data = train_data[(train_data['power'] > 0) & (train_data['power'] <= 600)]

# 车龄异常值处理
train_data = train_data[(train_data['car_age'] >= 0) & (train_data['car_age'] <= 50)]

# 价格异常值处理
Q1 = train_data['price'].quantile(0.25)
Q3 = train_data['price'].quantile(0.75)
IQR = Q3 - Q1
lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR
train_data = train_data[(train_data['price'] >= lower_bound) & (train_data['price'] <= upper_bound)]

print(f"异常值处理: 从 {original_size} 条记录中移除了 {original_size - len(train_data)} 条异常记录")


# 检查缺失值情况
missing_before = train_data.isnull().sum().sum()
print(f"处理前总缺失值数量: {missing_before}")

# notRepairedDamage用众数填充
if 'notRepairedDamage' in train_data.columns:
    damage_mode = train_data['notRepairedDamage'].mode()
    if not damage_mode.empty:
        damage_mode_value = damage_mode[0]
        train_data['notRepairedDamage'].fillna(damage_mode_value, inplace=True)
        test_data['notRepairedDamage'].fillna(damage_mode_value, inplace=True)
        print(f"notRepairedDamage缺失值用众数 {damage_mode_value} 填充")

# 数值特征用中位数填充
numerical_cols = ['power', 'kilometer', 'model', 'car_age'] + [f'v_{i}' for i in range(15)]
for col in numerical_cols:
    if col in train_data.columns:
        # 检查是否有缺失值
        if train_data[col].isnull().sum() > 0:
            train_median = train_data[col].median()
            train_data[col].fillna(train_median, inplace=True)
            if col in test_data.columns:
                test_data[col].fillna(train_median, inplace=True)
            print(f"{col}缺失值用中位数 {train_median:.2f} 填充")

# 检查处理后的缺失值
missing_after = train_data.isnull().sum().sum()
print(f"处理后总缺失值数量: {missing_after}")



# 品牌流行度
brand_popularity = train_data['brand'].value_counts().to_dict()
train_data['brand_popularity'] = train_data['brand'].map(brand_popularity)
test_data['brand_popularity'] = test_data['brand'].map(brand_popularity)

# 地区车辆密度
region_density = train_data['regionCode'].value_counts().to_dict()
train_data['region_density'] = train_data['regionCode'].map(region_density)
test_data['region_density'] = test_data['regionCode'].map(region_density)

# 车型流行度
model_popularity = train_data['model'].value_counts().to_dict()
train_data['model_popularity'] = train_data['model'].map(model_popularity)
test_data['model_popularity'] = test_data['model'].map(model_popularity)

# 功率分段
train_data['power_category'] = pd.cut(train_data['power'], 
                                     bins=[0, 100, 150, 200, 300, 600],
                                     labels=['低功率', '中低功率', '中等功率', '中高功率', '高功率'])
test_data['power_category'] = pd.cut(test_data['power'],
                                    bins=[0, 100, 150, 200, 300, 600],
                                    labels=['低功率', '中低功率', '中等功率', '中高功率', '高功率'])

# 里程分段
train_data['mileage_category'] = pd.cut(train_data['kilometer'],
                                       bins=[0, 5, 10, 15, 20],
                                       labels=['极低里程', '低里程', '中等里程', '高里程'])
test_data['mileage_category'] = pd.cut(test_data['kilometer'],
                                      bins=[0, 5, 10, 15, 20],
                                      labels=['极低里程', '低里程', '中等里程', '高里程'])



# 标准化的数值特征
features_to_scale = ['power', 'kilometer', 'model', 'car_age', 'brand_popularity', 
                    'region_density', 'model_popularity'] + [f'v_{i}' for i in range(15)]

# 存在的特征
features_to_scale = [f for f in features_to_scale if f in train_data.columns]

# 初始化标准化器
scaler = StandardScaler()

# 对训练集进行标准化
train_scaled = scaler.fit_transform(train_data[features_to_scale])
train_scaled_df = pd.DataFrame(train_scaled, columns=[f'{col}_scaled' for col in features_to_scale])
train_data = pd.concat([train_data, train_scaled_df], axis=1)

# 对测试集进行标准化（使用训练集的参数）
test_scaled = scaler.transform(test_data[features_to_scale])
test_scaled_df = pd.DataFrame(test_scaled, columns=[f'{col}_scaled' for col in features_to_scale])
test_data = pd.concat([test_data, test_scaled_df], axis=1)


categorical_features = ['bodyType', 'fuelType', 'gearbox', 'notRepairedDamage', 
                       'seller', 'offerType', 'power_category', 'mileage_category']

label_encoders = {}

for feature in categorical_features:
    if feature in train_data.columns:
        # 训练集编码
        le = LabelEncoder()
        train_data[f'{feature}_encoded'] = le.fit_transform(train_data[feature].astype(str))
        
        # 测试集编码
        # 处理测试集中可能出现的新类别
        test_data[f'{feature}_encoded'] = test_data[feature].astype(str)
        unique_train = set(le.classes_)
        unique_test = set(test_data[feature].astype(str).unique())
        
        # 对于测试集中的新类别，用最常见的类别替换
        new_categories = unique_test - unique_train
        if new_categories:
            most_common = train_data[feature].mode()[0] if not train_data[feature].mode().empty else 0
            test_data.loc[test_data[f'{feature}_encoded'].isin(new_categories), f'{feature}_encoded'] = str(most_common)
        
        test_data[f'{feature}_encoded'] = le.transform(test_data[f'{feature}_encoded'])
        label_encoders[feature] = le
        
        print(f"{feature} 编码完成")




# 提取v系列特征
v_features = [f'v_{i}' for i in range(15)]
v_features = [f for f in v_features if f in train_data.columns]

if v_features:
    # 确保v_features没有缺失值
    print("检查v_features缺失值...")
    for feature in v_features:
        if train_data[feature].isnull().sum() > 0:
            print(f"{feature} 仍有 {train_data[feature].isnull().sum()} 个缺失值，进行填充...")
            train_median = train_data[feature].median()
            train_data[feature].fillna(train_median, inplace=True)
            test_data[feature].fillna(train_median, inplace=True)
    
    # 再次检查确保没有缺失值
    v_missing_train = train_data[v_features].isnull().sum().sum()
    v_missing_test = test_data[v_features].isnull().sum().sum()
    print(f"v_features缺失值情况 - 训练集: {v_missing_train}, 测试集: {v_missing_test}")
    
    if v_missing_train == 0 and v_missing_test == 0:
        # 应用PCA降维
        pca = PCA(n_components=5)  # 降维到5个主成分
        v_pca_train = pca.fit_transform(train_data[v_features])
        v_pca_test = pca.transform(test_data[v_features])
        
        # 创建PCA特征列
        for i in range(5):
            train_data[f'v_pca_{i+1}'] = v_pca_train[:, i]
            test_data[f'v_pca_{i+1}'] = v_pca_test[:, i]
        
        # 解释方差比
        explained_variance = pca.explained_variance_ratio_
        print(f"PCA前5个主成分解释的方差比例: {explained_variance}")
        print(f"累计解释方差: {sum(explained_variance):.3f}")
    else:
        print("警告: v_features仍有缺失值，跳过PCA")
else:
    print("没有找到v_features")


plt.figure(figsize=(15, 10))

# 处理前后的价格分布对比
plt.subplot(2, 3, 1)
plt.hist(train_data['price'], bins=50, alpha=0.7, color='skyblue', edgecolor='black', label='处理后')
plt.title('异常值处理后的价格分布')
plt.xlabel('价格')
plt.ylabel('频次')
plt.grid(True, alpha=0.3)

# 处理前后的功率分布对比
plt.subplot(2, 3, 2)
plt.hist(train_data['power'], bins=50, alpha=0.7, color='lightgreen', edgecolor='black')
plt.title('异常值处理后的功率分布')
plt.xlabel('功率')
plt.ylabel('频次')
plt.grid(True, alpha=0.3)

# 车龄分布
plt.subplot(2, 3, 3)
plt.hist(train_data['car_age'].dropna(), bins=30, alpha=0.7, color='orange', edgecolor='black')
plt.title('车龄分布')
plt.xlabel('车龄(年)')
plt.ylabel('频次')
plt.grid(True, alpha=0.3)

# 品牌流行度分布
plt.subplot(2, 3, 4)
plt.hist(train_data['brand_popularity'], bins=30, alpha=0.7, color='purple', edgecolor='black')
plt.title('品牌流行度分布')
plt.xlabel('品牌流行度')
plt.ylabel('频次')
plt.grid(True, alpha=0.3)

# 功率分段分布
plt.subplot(2, 3, 5)
power_cat_counts = train_data['power_category'].value_counts()
plt.bar(power_cat_counts.index, power_cat_counts.values, alpha=0.7, color='red', edgecolor='black')
plt.title('功率分段分布')
plt.xlabel('功率分段')
plt.ylabel('数量')
plt.xticks(rotation=45)
plt.grid(True, alpha=0.3)

# 里程分段分布
plt.subplot(2, 3, 6)
mileage_cat_counts = train_data['mileage_category'].value_counts()
plt.bar(mileage_cat_counts.index, mileage_cat_counts.values, alpha=0.7, color='brown', edgecolor='black')
plt.title('里程分段分布')
plt.xlabel('里程分段')
plt.ylabel('数量')
plt.xticks(rotation=45)
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('预处理图片/preprocessing_results.png', dpi=300, bbox_inches='tight')
plt.show()


train_data.to_csv('train_data_processed.csv', index=False)
test_data.to_csv('test_data_processed.csv', index=False)

print(f"预处理完成!")
print(f"最终训练集形状: {train_data.shape}")
print(f"最终测试集形状: {test_data.shape}")
print(f"训练集列数: {train_data.shape[1]}")
print(f"测试集列数: {test_data.shape[1]}")

In [ ]:
#第三步：数据挖掘与建模

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split,KFold
from sklearn.linear_model import Ridge, Lasso, ElasticNet 
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report,confusion_matrix,mean_squared_error 
from sklearn.cluster import KMeans
from mlxtend.frequent_patterns import apriori
from mlxtend.frequent_patterns import association_rules
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
import os
import warnings
import xgboost as xgb

warnings.filterwarnings('ignore')

plt.rcParams['font.sans-serif'] = ['SimHei'] 
plt.rcParams['axes.unicode_minus'] = False 
os.makedirs('数据挖掘图片', exist_ok=True)

train_data = pd.read_csv('train_data_processed.csv')
test_data = pd.read_csv('test_data_processed.csv')

print(f"数据加载成功。训练集形状: {train_data.shape}, 测试集形状: {test_data.shape}")

train_data.fillna(-1, inplace=True) # 使用-1填充所有剩余的NaN，以便模型处理
test_data.fillna(-1, inplace=True)  # 测试集也进行填充

# 定义价格等级用于分类和关联规则
def get_price_class(p):
    if p <= 3000: return 0       # 低端
    elif p <= 7000: return 1     # 中端
    else: return 2               # 高端

train_data['price_class'] = train_data['price'].apply(get_price_class)
class_names = ['低端 (<3k)', '中端 (3k-7k)', '高端 (>7k)']
print("已定义价格等级：0-低端，1-中端，2-高端。")

# 定义用于建模的特征列
feature_cols = ['power_scaled', 'kilometer_scaled', 'car_age', 'brand_popularity', 
                'region_density', 'model_popularity',
                'bodyType_encoded', 'fuelType_encoded', 'gearbox_encoded', 
                'notRepairedDamage_encoded', 'seller_encoded', 'offerType_encoded'] + \
                [f'v_pca_{i+1}' for i in range(5)]
feature_cols = [f for f in feature_cols if f in train_data.columns]



#关联规则 (Apriori)
#创建包含离散特征的 DataFrame
apriori_df = train_data[['price_class', 'bodyType_encoded', 'gearbox_encoded', 
                         'power_category', 'mileage_category', 'fuelType_encoded']].copy()

# 将price_class转换为类别标签
apriori_df['price_category'] = apriori_df['price_class'].map({0: 'LowPrice', 1: 'MidPrice', 2: 'HighPrice'})
apriori_df.drop('price_class', axis=1, inplace=True)

# 转换为 One-Hot Encoding
for col in apriori_df.columns:
    apriori_df[col] = apriori_df[col].astype(str)

apriori_ohe = pd.get_dummies(apriori_df, prefix_sep='=').astype(bool)

# Apriori算法：生成频繁项集
min_support_value = 0.05 
frequent_itemsets = apriori(apriori_ohe, 
                            min_support=min_support_value, 
                            use_colnames=True)

# 生成关联规则置信度 > 0.6, 提升度 > 1.2
rules = association_rules(frequent_itemsets, 
                          metric="confidence", 
                          min_threshold=0.6)
rules = rules[(rules['lift'] >= 1.2)].sort_values(by=['lift', 'confidence'], ascending=False).reset_index(drop=True)

# 结果可视化
plt.figure(figsize=(10, 6))
scatter = plt.scatter(rules['support'], rules['confidence'], alpha=0.7, c=rules['lift'], cmap='viridis', s=rules['lift']*50)
cbar = plt.colorbar(scatter)
cbar.set_label('提升度 (Lift)', rotation=270, labelpad=15)
plt.title('关联规则支持度 vs. 置信度')
plt.xlabel('支持度 (Support)')
plt.ylabel('置信度 (Confidence)')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('数据挖掘图片/result_association_rules.png', dpi=300)

# 输出最具价值的规则
rules_to_show = rules[['antecedents', 'consequents', 'support', 'confidence', 'lift']].head(5)
rules_to_show['antecedents'] = rules_to_show['antecedents'].apply(lambda x: ', '.join(list(x)))
rules_to_show['consequents'] = rules_to_show['consequents'].apply(lambda x: ', '.join(list(x)))
print("\n高价值关联规则 (Top 5):\n", rules_to_show)



# 价格等级分类预测 (随机森林)
X = train_data[feature_cols]
y = train_data['price_class']

X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=2023, stratify=y)

# 训练随机森林模型
rf = RandomForestClassifier(n_estimators=100, max_depth=10, random_state=2023, class_weight='balanced')
rf.fit(X_train, y_train)

# 混淆矩阵结论可视化
y_pred = rf.predict(X_val)
plt.figure(figsize=(8, 6))
cm = confusion_matrix(y_val, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=class_names, yticklabels=class_names)
plt.title('二手车价格等级预测 - 混淆矩阵')
plt.xlabel('预测类别')
plt.ylabel('真实类别')
plt.tight_layout()
plt.savefig('数据挖掘图片/result_confusion_matrix.png', dpi=300)

# 特征重要性结论可视化
plt.figure(figsize=(10, 8))
imp = pd.Series(rf.feature_importances_, index=feature_cols).sort_values(ascending=False).head(15)
imp.plot(kind='barh', color=sns.color_palette("viridis", len(imp)))
plt.title('决定二手车价格等级的关键特征 (Top 15)')
plt.xlabel('特征重要性权重')
plt.tight_layout()
plt.savefig('数据挖掘图片/result_feature_importance1.png', dpi=300)
print("\n分类模型验证集性能报告:\n", classification_report(y_val, y_pred, target_names=class_names))



#聚类分析 (K-Means)
# 聚类特征使用标准化后的核心属性
cluster_cols = ['power_scaled', 'kilometer_scaled', 'car_age', 'model_popularity']

# 训练K-Means选择 4 类
kmeans = KMeans(n_clusters=4, random_state=2023, n_init=10)
train_data['cluster'] = kmeans.fit_predict(train_data[cluster_cols])

# 结果分析
cluster_summary = train_data.groupby('cluster')[['price', 'car_age', 'power', 'kilometer', 'brand_popularity']].mean()
print("\n聚类群体特征均值 :\n", cluster_summary)

# 对聚类特征进行降维到2维
X_cluster = train_data[cluster_cols]
pca_cluster = PCA(n_components=2)
X_pca_cluster = pca_cluster.fit_transform(X_cluster)

plt.figure(figsize=(10, 7))
scatter = plt.scatter(X_pca_cluster[:, 0], X_pca_cluster[:, 1], c=train_data['cluster'], 
                      cmap='tab10', alpha=0.7, s=15, edgecolors='w', linewidths=0.5)
plt.title('二手车市场细分聚类结果 (K-Means, PCA降维)')
plt.xlabel('主成分 1 (PC1)')
plt.ylabel('主成分 2 (PC2)')
plt.colorbar(scatter, label='聚类类别')
plt.legend(*scatter.legend_elements(), title="Cluster")
plt.tight_layout()
plt.savefig('数据挖掘图片/result_clustering.png', dpi=300)




#高级回归预测 (XGBoost)
train_data['log_price'] = np.log1p(train_data['price'])
y_reg = train_data['log_price'] # 用于回归的目标

finite_mask = np.isfinite(y_reg)
if not finite_mask.all():
    print(f"警告：检测到 {np.sum(~finite_mask)} 个非有限（NaN/Inf）目标值，已将特征和目标对应行移除。")
    X_reg_full = train_data[feature_cols].copy()
    X = X_reg_full.loc[finite_mask]
    y_reg = y_reg.loc[finite_mask]
    
# 重新划分数据集使用log_price作为目标
X_train, X_val, y_train_reg, y_val_reg = train_test_split(X, y_reg, test_size=0.2, random_state=2023)

# 训练基准模型(Ridge)
ridge = Ridge(alpha=1.0, random_state=2023)
ridge.fit(X_train, y_train_reg) 
ridge_pred = ridge.predict(X_val)
ridge_mse = mean_squared_error(y_val_reg, ridge_pred)
ridge_rmse = np.sqrt(ridge_mse)
print(f"Ridge 回归验证集 RMSE (log): {ridge_rmse:.4f}")

# 使用XGBoost进行预测
xgb = XGBRegressor(
    n_estimators=1000, # 使用预设的 1000 棵树
    learning_rate=0.05, 
    max_depth=7, 
    subsample=0.8, 
    colsample_bytree=0.8,
    random_state=2023,
    tree_method='hist' # 优化训练速度
)

# 拟合模型
xgb.fit(X_train, y_train_reg,
        eval_set=[(X_val, y_val_reg)], 
        verbose=False)

# 预测并反转对数变换
xgb_pred_log = xgb.predict(X_val)
xgb_pred = np.expm1(xgb_pred_log)
y_val_original = np.expm1(y_val_reg) # 从 log 反转回原始价格

# 计算原始价格的 RMSE均方根误差
def rmsel_score(y_true, y_pred):
    return np.sqrt(mean_squared_error(np.log1p(y_true), np.log1p(y_pred)))

final_rmse = rmsel_score(y_val_original, xgb_pred)
print(f"XGBoost 验证集 RMSE (log scale): {final_rmse:.4f}")
original_rmse = np.sqrt(mean_squared_error(y_val_original, xgb_pred))
print(f"XGBoost 验证集原始价格 RMSE: {original_rmse:.2f} 元")

# 实际值vs预测值 
plt.figure(figsize=(10, 6))
plt.scatter(y_val_original, xgb_pred, alpha=0.3, s=15, c='darkorange')
plt.plot([0, y_val_original.max()], [0, y_val_original.max()], 'k--', lw=2, label='理想预测线')
plt.title('XGBoost 回归预测值 vs. 真实值')
plt.xlabel('真实价格 (元)')
plt.ylabel('预测价格 (元)')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('数据挖掘图片/result_regression_scatter.png', dpi=300)

# 特征重要性
plt.figure(figsize=(10, 8))
importance = pd.Series(xgb.feature_importances_, index=X.columns).sort_values(ascending=False).head(15)
importance.plot(kind='barh', color=sns.color_palette("rocket", len(importance)))
plt.title('XGBoost 回归模型特征重要性 (Top 15)')
plt.xlabel('重要性得分')
plt.tight_layout()
plt.savefig('数据挖掘图片/result_regression_importance2.png', dpi=300)



# 测试集预测与结果填充
# 重新训练最终模型 (使用全部训练数据 X 和 y_reg)
if hasattr(xgb, 'best_ntree_limit'):
    n_estimators_final = xgb.best_ntree_limit
else:
    n_estimators_final = xgb.get_params()['n_estimators']
    
final_xgb = XGBRegressor(
    n_estimators=n_estimators_final,
    learning_rate=0.05, 
    max_depth=7, 
    subsample=0.8, 
    colsample_bytree=0.8,
    random_state=2023,
    tree_method='hist'
)
# 使用全部经过清洗的训练集进行最终拟合
final_xgb.fit(X, y_reg) 
print("XGBoost 最终模型已在全部训练集上完成拟合。")

# 准备测试集特征
X_test = test_data[feature_cols]

#预测并反转对数变换
test_pred_log = final_xgb.predict(X_test)
test_pred_price = np.expm1(test_pred_log)

# 生成最终文件
submission = pd.DataFrame({
    'SaleID': test_data['SaleID'],
    'price': test_pred_price.round(0) # 价格取整
})

# 检查是否存在负值
submission.loc[submission['price'] < 0, 'price'] = 0 
submission.to_csv('Final.csv', index=False)
print(f"预测结果摘要:\n{submission['price'].describe()}")